# Module 6.2: Direct Preference Optimization (DPO)

Welcome to the ultimate stage of modern foundation models!

In standard SFT (the previous notebook), we give the model a Prompt and a Perfect Answer. The model learns to replicate it point-blank.
However, human preference is messy. An answer like *"Sure, here is the code: ..."* is much more preferred over an answer like *"I can help you with that. The code you need is: ..."* because humans love brevity. How do we mathematically train for subjective "vibes" and safety?

## 1. The Death of RLHF (PPO)

Historically, OpenAI used **RLHF (Reinforcement Learning from Human Feedback)** via an algorithm called `PPO`. 
It required keeping *three* massive LLMs in memory simultaneously:
1. The Base Model (to keep track of baseline logits).
2. A Reward Model (to grade the answers).
3. The Active Training Model.

This was so complex and memory-intensive that only giant corporations could align models.

## 2. Enter DPO (Direct Preference Optimization)

In 2023, researchers proved mathematically that you can bypass the custom Reward model entirely. Instead, you just construct a dataset consisting of `(Prompt, Winning Answer, Losing Answer)`.

The math is elegantly simple:
1. Feed the `Prompt` + `Winning Answer` to the model. Grab its logit probability.
2. Feed the `Prompt` + `Losing Answer` to the model. Grab its logit probability.
3. Adjust the weights so the probability of the Winning Answer goes UP, and the probability of the Losing Answer goes DOWN recursively.

Let's simulate the core logic!

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss_simulation(policy_chosen_logps, policy_rejected_logps, reference_chosen_logps, reference_rejected_logps, beta=0.1):
    """
    A simplified PyTorch representation of the DPO math.
    beta: Controls how much we want to stick to the original base model (Reference Model).
    """
    # 1. How much does our ACTIVE model prefer the winner vs the loser?
    policy_diff = policy_chosen_logps - policy_rejected_logps
    
    # 2. How much did the BASE model prefer the winner vs the loser?
    reference_diff = reference_chosen_logps - reference_rejected_logps
    
    # 3. We calculate the relative ratio.
    # If our active model likes the winner MORE than the base model did, this ratio grows!
    ratios = policy_diff - reference_diff
    
    # 4. The magic DPO formula! We apply a smooth sigmoid function over the ratio
    # and try to minimize the negative log of it.
    # This pushes the model away from the rejected completion explicitly.
    loss = -F.logsigmoid(beta * ratios)
    return loss

# Mock some log probabilities (The model's confidence in the answers)
print("--- DPO SIMULATION ---")
# Active model loves the chosen answer (high logp), hates the rejected answer (low logp)
act_chosen = torch.tensor([-1.2])
act_rejected = torch.tensor([-3.4])

# Base model was indifferent
base_chosen = torch.tensor([-2.2])
base_rejected = torch.tensor([-2.1])

loss = dpo_loss_simulation(act_chosen, act_rejected, base_chosen, base_rejected)
print(f"Calculated DPO Loss: {loss.item():.4f}")
print("Because our active model already preferred the chosen answer, the loss is very low!")

## 🎉 THE END: You are a Transformer Master!

If you have followed from Module 1 linearly down to here, you have traversed the entire timeline of AI over the last 7 years.

From dot products and soft-maxes, 
To positional waves and Multi-Head Attention,
Through the autoregressive training loop and KV caching,
All the way up to Chat formatting and Preference Optimization.

You now know EXACTLY what is running on a GPU cluster when you prompt an LLM. Thank you for reading!
- The LLM Workout Team